⚡ 01 — Multithreading vs Multiprocessing: The Real Story of the Tea Stall

**Topic 12.01 | Async/Parallel Python — Part 1 of 4** *(THEORY + LIVE demos)*

🌍 **Hook:** How does ONE tea-stall owner handle 100 hungry customers?
There are three ways:
- **Sequential:** Make tea for each customer one by one — 100 × 1 min = 100 min 😭
- **Threading:** The same single person, but while the milk boils (the WAIT), he takes another customer's order —
  wait-time is also work-time! ☕⚡
- **Multiprocessing:** Hire 4 DIFFERENT tea-stall owners — each with their OWN utensils and their OWN stove.
  Expensive, but 4x work truly happens at the same time! 💪
- (Async? That's the superhero of the next topic — writes the order, forgets about it, and picks it up when the bell rings 🦸)

In this notebook: **the DNA-level difference between process vs thread, the truth about the GIL, IO-bound vs CPU-bound**
— all with LIVE timers!

## 🐣 Level 0 — Process vs Thread: The DNA Difference 🧬

📖 **Definitions:**
- **Process** = a separate PROGRAM — its own interpreter, its own memory, everything of its own (a separate tea-stall owner)
- **Thread** = a lightweight worker INSIDE that same program — SHARES MEMORY (the hands and feet of one tea-stall owner)

🔍 **WHY is a thread light and a process heavy?** A process = a full copy of the house (`memory MBs` — expensive);
a Thread = add an extra pair of hands in the same house (`KBs`) — fast to create, easy to share.

📊 **ASCII:**
```
 PROCESS (4 chaiwale)                 THREADS (1 chaiwala, 3 order-queues)
 ┌──┐┌──┐┌──┐┌──┐   alag memory      ┌─T1─┐┌─T2─┐┌─T3─┐  SAME memory!
 │🧠││🧠││🧠││🧠│   mehnga start    └──┬──┴──┬──┴──┬──┘
 └──┘└──┘└──┘└──┘   (safe-isolate!)     └───── shared ─────┘
```
⚠️ **Rule:** Sharing = fast communication but the danger of race conditions!
(Two hands on one tea-packet — where did the sugar double?!)

In [1]:
import os, threading

print("🧬 DNA-CHECK:")
print(f"   humara process PID : {os.getpid()}  (is ghar ka aadhar-card #)")
print(f"   active threads abhi: {threading.active_count()}  (shuru me bas MAIN hi)")
print(f"   main thread ka naam: {threading.current_thread().name}")

def worker(n):
    print(f"   ⚙️ thread-{n} reporting — naam: {threading.current_thread().name} | PID wahi: {os.getpid()}")

ts = [threading.Thread(target=worker, args=(i,), name=f"chaiwala-{i}") for i in range(3)]
for t in ts: t.start()
for t in ts: t.join()
print(f"\n   kaam khatam → active threads wapas: {threading.active_count()}")
print("   📌 NOTE: sabka PID SAME = threads ek HI process ke andar the (share-success!)")
# Expected OUTPUT (thread-lines ka ORDER har baar alag — yehi race-ka-maza!):
# 🧬 DNA-CHECK:
#    humara process PID : XXXX  ...
#    active threads abhi: 1
#    main thread: MainThread
#    ⚙️ thread-0 reporting — naam: chaiwala-0 | PID wahi: XXXX
#    ⚙️ thread-1 ... / thread-2 ...
#    kaam khatam → active threads wapas: 1

🧬 DNA-CHECK:
   humara process PID : 6573  (is ghar ka aadhar-card #)
   active threads abhi: 1  (shuru me bas MAIN hi)
   main thread ka naam: MainThread
   ⚙️ thread-0 reporting — naam: chaiwala-0 | PID wahi: 6573
   ⚙️ thread-1 reporting — naam: chaiwala-1 | PID wahi: 6573
   ⚙️ thread-2 reporting — naam: chaiwala-2 | PID wahi: 6573

   kaam khatam → active threads wapas: 1
   📌 NOTE: sabka PID SAME = threads ek HI process ke andar the (share-success!)


## 🐥 Level 1 — IO-BOUND vs CPU-BOUND: Which Task, Which Kind of Exhaustion? 🏋️

📖 **Definitions:**
- **IO-bound** = the task involves LITTLE WORK but **LOTS OF WAIT** (file read, network call, DB query, sleep)
- **CPU-bound** = the task involves **LOTS OF THINKING** (math crunch, image processing, ML training)

📊 **Decision table (remember exactly this one!):**
| Nature of the task | Champion | Why |
|---|---|---|
| IO-bound (waiting-game) | **Threading / Async** | CPU is released during wait → other work gets done |
| CPU-bound (thinking-game) | **Multiprocessing** | separate core = truly parallel compute |

Today's IO experiment: 3 "file-reads" (each a 0.3s wait) — sequential vs threaded!

In [2]:
import time

def fake_file_read(n, secs=0.3):
    """0.3s ka WAIT — maano bade CSV load ka time (IO-bound kaan!)"""
    time.sleep(secs)
    return f"file-{n}-done"

# (a) SEQUENTIAL: 3 files ek-ek karke
t0 = time.perf_counter()
res_seq = [fake_file_read(i) for i in range(3)]
seq_t = time.perf_counter() - t0
print(f"🐢 SEQUENTIAL: {res_seq} | time = {seq_t:.2f}s (3×0.3 ≈ 0.9)")

# (b) THREADED: 3 files EK-SAATH wait me
t0 = time.perf_counter()
results = [None] * 3
def worker(i):
    results[i] = fake_file_read(i)
ts = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
for t in ts: t.start()
for t in ts: t.join()
thr_t = time.perf_counter() - t0
print(f"⚡ THREADED   : {results} | time = {thr_t:.2f}s (sab overlap ≈ 0.3)")
print(f"\n🏁 SPEEDUP: {seq_t/thr_t:.1f}x — wait-time=kaam-time ka inaam! ☕")
# Expected OUTPUT (timings thodi upar-neeche — concept hi number hai!):
# 🐢 SEQUENTIAL: ['file-0-done', ...] | time = 0.90s
# ⚡ THREADED   : ['file-0-done', ...] | time = 0.30s
# 🏁 SPEEDUP: ~3.0x — wait-time=kaam-time ka inaam! ☕

🐢 SEQUENTIAL: ['file-0-done', 'file-1-done', 'file-2-done'] | time = 0.90s (3×0.3 ≈ 0.9)
⚡ THREADED   : ['file-0-done', 'file-1-done', 'file-2-done'] | time = 0.30s (sab overlap ≈ 0.3)

🏁 SPEEDUP: 3.0x — wait-time=kaam-time ka inaam! ☕


## 🐔 Level 2 — GIL: Python's Single-Key Lock 🔑

📖 **Definition:** In CPython the **GIL** (Global Interpreter Lock) is one big global
lock — at any moment ONLY ONE thread can run Python bytecode. Even with 100 threads,
CPU-heavy work waits in line behind it (a gents' toilet with a single door)!

🔍 **WHY does IO threading still work then?** During time.sleep / file-read / network call the
thread **releases** the GIL (it hangs up the key while waiting!). During heavy CPU work there
is NO release — everyone queues up behind the key.

Today's GIL-proof: 2 CPU-crunching threads — the speedup should be ZERO!

In [3]:
def crunch(n, loops=4_000_000):
    """pure CPU-soch — koi wait nahi, sirf ginti!"""
    s = 0
    for i in range(loops):
        s += i * i
    return f"crunch-{n}"

# (a) sequential 2 crunches
t0 = time.perf_counter()
_ = crunch(0); _ = crunch(1)
s_t = time.perf_counter() - t0

# (b) 2 threaded crunches — GIL bolta "ek-ek hi karoge! 🔑"
t0 = time.perf_counter()
res = [None] * 2
def cw(i): res[i] = crunch(i)
ts = [threading.Thread(target=cw, args=(i,)) for i in range(2)]
for t in ts: t.start()
for t in ts: t.join()
th_t = time.perf_counter() - t0

print(f"🐢 CPU-crunch SEQUENTIAL: {s_t:.2f}s")
print(f"🔒 CPU-crunch THREADED  : {th_t:.2f}s  ← ~SAME ya THODA worse (switching-cost!)")
print(f"   speedup = {s_t/th_t:.2f}x ≈ 1.0 → GIL ka LIVE pramaan! 🔑")
print("   ☝️ IO me 3x, CPU me 1x — isi liye kaam pehchaan ke hathiyar chuno!")
# Expected OUTPUT (ratios machine pe ~0.8-1.1 — GIL-point hi maarmik!):
# 🐢 CPU-crunch SEQUENTIAL: ~1.2s
# 🔒 CPU-crunch THREADED  : ~1.2s  ← ~SAME
#    speedup = ~1.0x ≈ 1.0 → GIL ka LIVE pramaan! 🔑

🐢 CPU-crunch SEQUENTIAL: 0.39s
🔒 CPU-crunch THREADED  : 0.37s  ← ~SAME ya THODA worse (switching-cost!)
   speedup = 1.06x ≈ 1.0 → GIL ka LIVE pramaan! 🔑
   ☝️ IO me 3x, CPU me 1x — isi liye kaam pehchaan ke hathiyar chuno!


## 🐔 Level 3 — Multiprocessing: Bring in 4 Tea-Stall Owners! 💪

📖 **Definition:** `multiprocessing` means spawning brand-new PYTHON PROCESSES —
each with its own interpreter and memory, and EACH has its own separate GIL! Run CPU-crunch
work on different cores in truly-parallel fashion.

⚠️ **3 Gotchas (that trip everyone up!):**
1. **The `if __name__ == "__main__":` guard is a MUST** — new processes import the whole
   file, and without the guard you get an ∞-spawn loop 💥
2. **Functions must be picklable** (module top-level) — lambdas/nested ❌
3. Multiprocessing is **flakey** in Jupyter/notebook execution — run it in a `.py` script (blocking work!)

Below is a real sample — **run it in a `.py` file on your machine** (a demo-sample here):

In [4]:
SAMPLE = '''
from multiprocessing import Pool
import time

def crunch(n):                      # top-level = picklable ✅
    return sum(i * i for i in range(4_000_000))

if __name__ == "__main__":          # GUARD — bhoologe toh spawn-bomb 💥
    t0 = time.perf_counter()
    with Pool(4) as p:              # 4 chaiwale = 4 alag processes 💪
        res = p.map(crunch, range(4))
    print("4-processes time:", round(time.perf_counter() - t0, 2), "s")
    # 4.8s ka kaam ≈ 1.3s me — TRULY parallel (har process apna GIL!)
'''
print("📝 .py-file SAMPLE (local me chalake dekho — yahan honest-note ke saath):")
print(SAMPLE)
print("🔑 RECAP-TABLE:")
print("   IO-bound  → threading/async  (wait = kaam!)")
print("   CPU-bound → multiprocessing  (alag core = sach-ka-parallel)")
print("   GIL       → sirf CPU-threads ko rokta, IO-threads free!")
# Expected OUTPUT:
# 📝 .py-file SAMPLE ...
# from multiprocessing import Pool
# ... (poora sample code)
# 🔑 RECAP-TABLE:
#    IO-bound  → threading/async  (wait = kaam!)
#    CPU-bound → multiprocessing  (alag core = sach-ka-parallel)
#    GIL       → sirf CPU-threads ko rokta, IO-threads free!

📝 .py-file SAMPLE (local me chalake dekho — yahan honest-note ke saath):

from multiprocessing import Pool
import time

def crunch(n):                      # top-level = picklable ✅
    return sum(i * i for i in range(4_000_000))

if __name__ == "__main__":          # GUARD — bhoologe toh spawn-bomb 💥
    t0 = time.perf_counter()
    with Pool(4) as p:              # 4 chaiwale = 4 alag processes 💪
        res = p.map(crunch, range(4))
    print("4-processes time:", round(time.perf_counter() - t0, 2), "s")
    # 4.8s ka kaam ≈ 1.3s me — TRULY parallel (har process apna GIL!)

🔑 RECAP-TABLE:
   IO-bound  → threading/async  (wait = kaam!)
   CPU-bound → multiprocessing  (alag core = sach-ka-parallel)
   GIL       → sirf CPU-threads ko rokta, IO-threads free!


---
## ⚠️ 5 Classic Mistakes (concurrency edition)
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Threads for CPU-crunch (GIL-lockdown!) | if CPU-bound → multiprocessing |
| 2 | Forgetting the `__main__` guard in mp (spawn-bomb) | always write the guard |
| 3 | lambda/nested fn in Pool.map (pickle-fail) | top-level def only |
| 4 | Mutating a shared list without a lock (race!) | Lock/Queue or the result-collect pattern |
| 5 | threads count = task count always (1000-threads overhead) | tune max_workers in ThreadPoolExecutor |

## 📌 Recap — 30 Seconds
- Process = separate house (own memory, expensive) · Thread = a hand in that house (shared!) 🏠
- IO-bound + threads = waiting-time jackpot (0.9s→0.3s LIVE!) ⚡
- GIL = a single key that queues up CPU threads 🔒 (IO releases it, free!)
- Multiprocessing = separate interpreters → CPU-crunch truly parallel 💪 (guard+pickle)

**Next:** `02_comparison_demo.ipynb` — a 10-task RACE-program! Sequential vs threaded
bar-race, live GIL-plateau, full timing scorecard 🏁